# Model Evaluation for Deployment

This notebook evaluates a candidate model version before deployment.

**Purpose:**
- Load model from Unity Catalog
- Evaluate on validation/test dataset
- Calculate MAP@12 and other metrics
- Log evaluation results to MLflow

**Note:** This notebook should only be run in a Databricks Job, as part of MLflow 3.0 Deployment Jobs.

## Setup

In [0]:
import sys

# Add project root to path (go up 2 levels from deployment_step/)
sys.path.append("../../")

from pyspark.sql.functions import *
import mlflow

from config.catalog_config import get_table_config
from utils.data_utils import load_delta_table
from utils.evaluation_utils import calculate_map_at_k, log_evaluation_metrics

In [0]:
# Define widgets for job parameters
dbutils.widgets.text("model_name", "")
dbutils.widgets.text("model_version", "")
dbutils.widgets.text("catalog_name", "shared")
dbutils.widgets.text("schema_name", "fashion_recommendations")
dbutils.widgets.text("evaluation_dataset", "val")  # 'val' or 'test'

In [0]:
# Get parameters
model_name = dbutils.widgets.get("model_name")
model_version = dbutils.widgets.get("model_version")
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
evaluation_dataset = dbutils.widgets.get("evaluation_dataset")

print(f"Evaluating Model: {model_name}")
print(f"Version: {model_version}")
print(f"Catalog: {catalog_name}")
print(f"Schema: {schema_name}")
print(f"Evaluation Dataset: {evaluation_dataset}")

## Load Evaluation Data

In [0]:
# Get table names
tables = get_table_config(catalog_name, schema_name)

# Load ground truth based on evaluation dataset parameter
if evaluation_dataset == "test":
    ground_truth_table = tables.TEST_GROUND_TRUTH_SILVER
    print(f"Using TEST dataset: {ground_truth_table}")
else:
    ground_truth_table = tables.VAL_GROUND_TRUTH_SILVER
    print(f"Using VALIDATION dataset: {ground_truth_table}")

# Load ground truth
ground_truth_df = load_delta_table(ground_truth_table)
print(f"Loaded ground truth: {ground_truth_df.count():,} customers")

# Load customers for context (optional)
customers_df = load_delta_table(tables.CUSTOMERS_BRONZE)
print(f"Total customers: {customers_df.count():,}")

## Load Model and Generate Predictions

In [0]:
# Load model from Unity Catalog
model_uri = f"models:/{model_name}/{model_version}"
print(f"Loading model: {model_uri}")

# Load model using MLflow
model = mlflow.pyfunc.load_model(model_uri)
print("Model loaded successfully")

In [0]:
# Generate predictions for evaluation customers
print("Generating predictions...")

# Get customer IDs from ground truth
eval_customers = ground_truth_df.select("customer_id").toPandas()

# Predict using model
predictions_pd = model.predict(eval_customers)

# Convert to Spark DataFrame
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

predictions_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("predicted_articles", ArrayType(StringType()), False)
])

predictions_df = spark.createDataFrame(predictions_pd, schema=predictions_schema)
print(f"Generated predictions: {predictions_df.count():,} customers")
display(predictions_df.limit(5))

## Evaluate Model Performance

In [0]:
# Start MLflow run for evaluation
with mlflow.start_run(run_name="deployment_evaluation") as run:
    # Log model info
    mlflow.set_tag("model_name", model_name)
    mlflow.set_tag("model_version", model_version)
    mlflow.set_tag("evaluation_dataset", evaluation_dataset)
    mlflow.set_tag("stage", "deployment_evaluation")
    
    # Calculate and log metrics
    print("\n" + "="*60)
    print(f"EVALUATING {model_name} v{model_version}")
    print("="*60)
    
    metrics = log_evaluation_metrics(
        predictions_df,
        ground_truth_df,
        model_name=f"{model_name} v{model_version}",
        k=12
    )
    
    # Save run ID
    evaluation_run_id = run.info.run_id
    print(f"\nEvaluation MLflow run ID: {evaluation_run_id}")
    print(f"MAP@12: {metrics['map@12']:.6f}")

## Summary

In [0]:
print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)
print(f"Model: {model_name}")
print(f"Version: {model_version}")
print(f"MAP@12: {metrics['map@12']:.6f}")
print(f"Evaluated on: {evaluation_dataset} dataset")
print(f"Customers evaluated: {metrics['num_customers']:,}")
print("="*60)
print("\n✓ Model evaluation successful. Ready for approval step.")